# Low-level Evaluation on Augmented Data

## This is Part 4.2. in the official paper, image synthesis method: CycleGAN

We use official PyTorch implementation of CycleGAN: https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix

We did experiments on 4 domains:
1. Chocolate cake <-> Vanilla cake
2. Zebra <-> Horse
3. Orange <-> Apple
4. Milk <-> Bubble Milk

Images were initially resized to 288x288 and stored in different folders per domain because of the model's requirements. Each domain specific folder contains folders 'trainA' and 'testA' - for content images and 'trainB' and 'testB' - for style images.

In [ ]:
!pip install torch torchvision matplotlib numpy scikit-learn

In [1]:
!pip install torchmetrics clean-fid
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.9/960.9 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time

from PIL import Image
import matplotlib.pyplot as plt

from cleanfid import fid
import numpy as np

import lpips
import os
import sys
import random


import torchvision.transforms as transforms
from torchvision.models import vgg19, VGG19_Weights

import copy

In [3]:
# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_COLAB

True

In [4]:
# Set project path based on the environment
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/My Drive"):
      drive.mount('/content/drive')
    else:
      print("Drive already mounted")

    project_path = "/content/drive/My Drive/4.2. CycleGAN/"
else:
    project_path = "../"

os.chdir(project_path)
project_path

Mounted at /content/drive


'/content/drive/My Drive/4.2. CycleGAN/'

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
device

device(type='cuda')

In [ ]:
!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix


Cloning into 'pytorch-CycleGAN-and-pix2pix'...
remote: Enumerating objects: 2516, done.
remote: Total 2516 (delta 0), reused 0 (delta 0), pack-reused 2516 (from 1)
Receiving objects: 100% (2516/2516), 8.20 MiB | 18.42 MiB/s, done.
Resolving deltas: 100% (1575/1575), done.
/content/drive/MyDrive/4.2. CycleGAN/pytorch-CycleGAN-and-pix2pix
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408196 sha256=74413847b0fe134a85d8489236d13c55ad9f375059875086bedfcabd852373c7
  Stored in directory: /root/.cache/pip/wheels/fa/a4/bb/2be445c295d88a74f9c0a4232f04860ca489a5c7c57eb959d9
Successfully built visdom


In [6]:
%cd pytorch-CycleGAN-and-pix2pix

/content/drive/.shortcut-targets-by-id/1U1nknySqWANU5JGLf-mVvR4Z-aPCcnsE/4.2. CycleGAN/pytorch-CycleGAN-and-pix2pix


In [15]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 39.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for visdom: filename=visdom-0.2.4-py3-none-any.whl size=1408196 sha256=e8a10bba20cf9fae17c7f68398cca98761002a8af50c74e8e7354f0b5eced94d
  Stored in directory: /root/.cache/pip/wheels/fa/a4/bb/2be445c295d88a74f9c0a4232f04860ca489a5c7c57eb959d9
Successfully built visdom


## Defining metrics:

*   Fréchet Inception Distance (FID)

*   Learned Perceptual Image Patch Similarity (LPIPS)

*   VGG Loss (perceptual loss)


In [7]:
def calculate_fid(image1_path: str, image2_path: str) -> float:
    # image1_path: Path to generated image directory
    # image2_path: Path to target style image directory

    fid_value = fid.compute_fid(image1_path, image2_path)
    return fid_value


lpips_loss = lpips.LPIPS(net='vgg').to(device)


def load_image(image_path):
    transform = transforms.Compose([
        transforms.Resize((288, 288)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    image = Image.open(image_path).convert("RGB")
    return transform(image).unsqueeze(0).to(device)

def calculate_lpips(image1_path: str, image2_path: str) -> float:
    # Load images and preprocess
    img1 = load_image(image1_path).to(device)
    img2 = load_image(image2_path).to(device)

    with torch.no_grad():
        distance = lpips_loss(img1, img2)

    return distance.item()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:04<00:00, 134MB/s]


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


In [8]:
vgg_model = vgg19(pretrained=True).features[:8].eval().to(device)
loss_fn = torch.nn.MSELoss()

def calculate_vgg_loss(real_image, generated_image):
    img1 = load_image(real_image)
    img2 = load_image(generated_image)
    with torch.no_grad():
        f1 = vgg_model(img1)
        f2 = vgg_model(img2)
        return loss_fn(f1, f2).item()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:04<00:00, 126MB/s]


# Experiment 1: Vanilla cake <-> Chocolate cake

We use single source and target image. We apply an augmentation pipeline to the input images such as random crops or flips and increase the number of input training images.

In [9]:
def augment_image(image_path, output_folder, num_augmented_images=99):
    os.makedirs(output_folder, exist_ok=True)

    # Load the original image
    image = Image.open(image_path).convert('RGB')

    image = image.resize((288, 288))

    # Define augmentation pipeline
    augmentations = transforms.Compose([
        transforms.RandomResizedCrop(288, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(30),
    ])

    for i in range(num_augmented_images):
        augmented_image = augmentations(image)
        augmented_image.save(os.path.join(output_folder, f"aug_{i}.jpg"))

    print(f"Saved {num_augmented_images} images to {output_folder}")

In [ ]:
chocolate_cake_path =  os.path.join(project_path, "images/Cakes/trainB/ChocolateCake.jpg")
vanilla_cake_path = os.path.join(project_path, "images/Cakes/trainA/VanillaCake.jpg")

trainA_folder =  os.path.join(project_path, "images/Cakes/trainA/")
trainB_folder =  os.path.join(project_path, "images/Cakes/trainB/")

# Generate augmented images
augment_image(vanilla_cake_path, trainA_folder)
augment_image(chocolate_cake_path, trainB_folder)

Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Cakes/trainA/
Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Cakes/trainB/


In [20]:
data_path = os.path.join(project_path, "images/Cakes")
data_path

'/content/drive/My Drive/4.2. CycleGAN/images/Cakes'

In the paper, the experiment was done on 800 iterations, which makes 8 epochs in this case (since batch_size = 1 and number_of_images = 100), However the results were not satisfying enough, so we increased the number of epochs

In [22]:
!python train.py --dataroot "$data_path" --name cakes_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB --n_epochs 20 --n_epochs_decay 0 --lr 0.0005 --num_threads 2 --serial_batches --display_id 0 --save_epoch_freq 5


----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Cakes	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
              display_env: main                          
             display_freq: 400                           
               display_id: 0                             	[default: 1]
            display_ncols: 4                             
             display_port: 8097                          
           display_server: http://localhost              
          display_winsize: 256                           
                    epoch: latest      

In [23]:
!time python test.py --dataroot "$data_path" --name cakes_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB

----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Cakes	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                  gpu_ids: 0                             
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 3                             
                  isTrain: False                         	[default: None]
                load_iter: 0        

In [25]:
output_file_path = os.path.join(os.getcwd(), "results/cakes_cycleGAN/test_latest/images/VanillaCake_fake_B.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Cakes/trainB/ChocolateCake.jpg")
print(f"LPIPS Score (Vanilla cake -> Chocolate Cake): {lpips_value}")


LPIPS Score (Vanilla cake -> Chocolate Cake): 0.4210321009159088


In [26]:
vgg_value = calculate_vgg_loss(project_path + "images/Cakes/trainA/VanillaCake.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 9.760967254638672


In [27]:
output_file_path = os.path.join(os.getcwd(), "results/cakes_cycleGAN/test_latest/images/VanillaCake_fake_A.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Cakes/trainA/VanillaCake.jpg")
print(f"LPIPS Score (Chocolate Cake -> Vanilla cake): {lpips_value}")

LPIPS Score (Chocolate Cake -> Vanilla cake): 0.38146141171455383


In [28]:
vgg_value = calculate_vgg_loss(project_path + "images/Cakes/trainB/ChocolateCake.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 11.288480758666992


# Experiment 2: Milk <-> Bubble Milk

In [ ]:
bubble_milk_path = os.path.join(project_path, "images/Milk/trainB/BubbleMilk.jpg")
milk_path = os.path.join(project_path, "images/Milk/trainA/Milk.jpg")

trainA_folder = os.path.join(project_path, "images/Milk/trainA/")
trainB_folder = os.path.join(project_path, "images/Milk/trainB/")

# Generate augmented images
augment_image(milk_path, trainA_folder)
augment_image(bubble_milk_path, trainB_folder)

Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Milk/trainA/
Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Milk/trainB/


In [30]:
data_path = os.path.join(project_path, "images/Milk")
data_path

'/content/drive/My Drive/4.2. CycleGAN/images/Milk'

In [31]:
!python train.py --dataroot "$data_path" --name milk_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB --n_epochs 20 --n_epochs_decay 0 --lr 0.0005 --num_threads 2 --serial_batches --display_id 0 --save_epoch_freq 5

----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Milk	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
              display_env: main                          
             display_freq: 400                           
               display_id: 0                             	[default: 1]
            display_ncols: 4                             
             display_port: 8097                          
           display_server: http://localhost              
          display_winsize: 256                           
                    epoch: latest       

In [32]:
!time python test.py --dataroot "$data_path" --name milk_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB

----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Milk	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                  gpu_ids: 0                             
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 3                             
                  isTrain: False                         	[default: None]
                load_iter: 0         

In [33]:
output_file_path = os.path.join(os.getcwd(), "results/milk_cycleGAN/test_latest/images/Milk_fake_B.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Milk/trainB/BubbleMilk.jpg")
print(f"LPIPS Score (Milk -> Bubble Milk): {lpips_value}")


LPIPS Score (Milk -> Bubble Milk): 0.5991061925888062


In [34]:
vgg_value = calculate_vgg_loss(project_path + "images/Milk/trainA/Milk.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 7.709981918334961


In [35]:
output_file_path = os.path.join(os.getcwd(), "results/milk_cycleGAN/test_latest/images/Milk_fake_A.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Milk/trainA/Milk.jpg")
print(f"LPIPS Score (Bubble milk -> Milk): {lpips_value}")


LPIPS Score (Bubble milk -> Milk): 0.36170831322669983


In [36]:
vgg_value = calculate_vgg_loss(project_path + "images/Milk/trainB/BubbleMilk.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 10.005791664123535


# Experiment 3: Zebra <-> Horse

In [37]:
zebra_path =  os.path.join(project_path, "images/Animals/trainB/Zebra.jpg")
horse_path = os.path.join(project_path, "images/Animals/trainA/Horse.jpg")

trainA_folder =  os.path.join(project_path, "images/Animals/trainA/")
trainB_folder =  os.path.join(project_path, "images/Animals/trainB/")

# Generate augmented images
augment_image(horse_path, trainA_folder)
augment_image(zebra_path, trainB_folder)

Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Animals/trainA/
Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Animals/trainB/


In [40]:
data_path = os.path.join(project_path, "images/Animals")
data_path

'/content/drive/My Drive/4.2. CycleGAN/images/Animals'

In [41]:
!python train.py --dataroot "$data_path" --name animal_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB --n_epochs 20 --n_epochs_decay 0 --lr 0.0005 --num_threads 2 --serial_batches --display_id 0 --save_epoch_freq 5

----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Animals	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
              display_env: main                          
             display_freq: 400                           
               display_id: 0                             	[default: 1]
            display_ncols: 4                             
             display_port: 8097                          
           display_server: http://localhost              
          display_winsize: 256                           
                    epoch: latest    

In [42]:
!time python test.py --dataroot "$data_path" --name animal_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB

----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Animals	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                  gpu_ids: 0                             
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 3                             
                  isTrain: False                         	[default: None]
                load_iter: 0      

In [44]:
output_file_path = os.path.join(os.getcwd(), "results/animal_cycleGAN/test_latest/images/Horse_fake_B.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Animals/trainB/Zebra.jpg")
print(f"LPIPS Score (Horse -> Zebra): {lpips_value}")


LPIPS Score (Horse -> Zebra): 0.7211266160011292


In [45]:
vgg_value = calculate_vgg_loss(project_path + "images/Animals/trainA/Horse.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 17.56613540649414


In [46]:
output_file_path = os.path.join(os.getcwd(), "results/animal_cycleGAN/test_latest/images/Horse_fake_A.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Animals/trainA/Horse.jpg")
print(f"LPIPS Score (Zebra -> Horse): {lpips_value}")

LPIPS Score (Zebra -> Horse): 0.7560234665870667


In [47]:
vgg_value = calculate_vgg_loss(project_path + "images/Animals/trainB/Zebra.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 19.935800552368164


# Experiment 4: Apple <-> Orange

In [48]:
orange_path =  os.path.join(project_path, "images/Fruits/trainB/Orange.jpg")
apple_path = os.path.join(project_path, "images/Fruits/trainA/Apple.jpg")

trainA_folder =  os.path.join(project_path, "images/Fruits/trainA/")
trainB_folder =  os.path.join(project_path, "images/Fruits/trainB/")

# Generate augmented images
augment_image(apple_path, trainA_folder)
augment_image(orange_path, trainB_folder)

Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Fruits/trainA/
Saved 99 images to /content/drive/My Drive/4.2. CycleGAN/images/Fruits/trainB/


In [50]:
data_path = os.path.join(project_path, "images/Fruits")
data_path

'/content/drive/My Drive/4.2. CycleGAN/images/Fruits'

In [51]:
!python train.py --dataroot "$data_path" --name fruit_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB --n_epochs 20 --n_epochs_decay 0 --lr 0.0005 --num_threads 2 --serial_batches --display_id 0 --save_epoch_freq 5

----------------- Options ---------------
               batch_size: 1                             
                    beta1: 0.5                           
          checkpoints_dir: ./checkpoints                 
           continue_train: False                         
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Fruits	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
              display_env: main                          
             display_freq: 400                           
               display_id: 0                             	[default: 1]
            display_ncols: 4                             
             display_port: 8097                          
           display_server: http://localhost              
          display_winsize: 256                           
                    epoch: latest     

In [52]:
!time python test.py --dataroot "$data_path" --name fruit_cycleGAN --model cycle_gan --netG resnet_9blocks --netD basic --direction AtoB

----------------- Options ---------------
             aspect_ratio: 1.0                           
               batch_size: 1                             
          checkpoints_dir: ./checkpoints                 
                crop_size: 256                           
                 dataroot: /content/drive/My Drive/4.2. CycleGAN/images/Fruits	[default: None]
             dataset_mode: unaligned                     
                direction: AtoB                          
          display_winsize: 256                           
                    epoch: latest                        
                     eval: False                         
                  gpu_ids: 0                             
                init_gain: 0.02                          
                init_type: normal                        
                 input_nc: 3                             
                  isTrain: False                         	[default: None]
                load_iter: 0       

In [53]:
output_file_path = os.path.join(os.getcwd(), "results/fruit_cycleGAN/test_latest/images/Apple_fake_B.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Fruits/trainB/Orange.jpg")
print(f"LPIPS Score (Apple -> Orange): {lpips_value}")


LPIPS Score (Apple -> Orange): 0.5324077010154724


In [54]:
vgg_value = calculate_vgg_loss(project_path + "images/Fruits/trainA/Apple.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 16.980817794799805


In [55]:
output_file_path = os.path.join(os.getcwd(), "results/fruit_cycleGAN/test_latest/images/Apple_fake_A.png")
lpips_value = calculate_lpips(output_file_path, project_path + "images/Fruits/trainA/Apple.jpg")
print(f"LPIPS Score (Orange -> Apple): {lpips_value}")

LPIPS Score (Orange -> Apple): 0.6087704300880432


In [56]:
vgg_value = calculate_vgg_loss(project_path + "images/Fruits/trainB/Orange.jpg", output_file_path)
print(f"VGG Loss: {vgg_value}")

VGG Loss: 11.560053825378418


#FID calculation

For FID calculation, we organized images in folders: augmented data and original images

In [57]:
fid_value = calculate_fid(project_path + "images/Results", project_path + "images/Original images")
print(f"FID Score: {fid_value}")

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


compute FID between two folders
Found 8 images in the folder /content/drive/My Drive/4.2. CycleGAN/images/Results


FID Results : 100%|██████████| 1/1 [00:03<00:00,  3.80s/it]


Found 8 images in the folder /content/drive/My Drive/4.2. CycleGAN/images/Original images


FID Original images : 100%|██████████| 1/1 [00:08<00:00,  8.51s/it]


FID Score: 305.07885301973
